# Задание 4.2. Как провисает нить между опорами
**Зелёный уровень**

Рассматривается однородная тяжёлая гибкая нерастяжимая нить. Опоры расположены в точках $(0,0)$ и $(1,0)$, ось $y$ направлена вверх. Требуется методом стрельбы решить краевую задачу
$$
y''=\frac{1}{a}\sqrt{1+(y')^2},\qquad
a=1,\qquad y(0)=y(1)=0,
$$
сравнить результат с точным решением и проверить точность при уменьшении шага.

Здесь $a=H/q$, где $H$ — горизонтальная составляющая натяжения, а $q$ — вес единицы длины нити. **По условию все длины выражены в единицах горизонтального расстояния между опорами.** Поэтому $x$, $y$, $a$ и шаг $h$ в расчёте — безразмерные численные значения; $h$ является шагом по координате $x$, а не по времени. Начальный наклон $s=y'(0)$ — отношение вертикального приращения к горизонтальному. Отрицательный $s$ означает, что от левой опоры нить сначала идёт вниз.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

a = 1.0
shoot_tol = 1e-14
counts = np.array([10, 20, 40, 80, 160])
colors = ["#b44e58", "#cf8c35", "#589465", "#397d9b", "#7458a0"]

## 1. Сведение к задаче Коши

Введём $p=y'$ и перепишем уравнение как систему:
$$
\begin{cases}
y'=p,\\
p'=\dfrac{\sqrt{1+p^2}}{a},
\end{cases}
\qquad
y(0)=0,\quad p(0)=s.
$$

При фиксированном начальном наклоне $s$ эту систему интегрируем от $x=0$ до $x=1$ классическим методом Рунге–Кутты четвёртого порядка (RK4). Для вектора $u=(y,p)$ и правой части $f(u)$ шаг имеет вид
$$
\begin{aligned}
k_1&=f(u_n),&
k_2&=f(u_n+hk_1/2),\\
k_3&=f(u_n+hk_2/2),&
k_4&=f(u_n+hk_3),\\
u_{n+1}&=u_n+\frac{h}{6}(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$

Используем $N$ равных шагов: $h=1/N$. Сетка содержит $N+1$ узел, включая обе опоры.

In [ ]:
def rhs(state, a):
    y, p = state
    return np.array([p, np.sqrt(1.0 + p*p)/a])


def integrate(s, n, a=1.0):
    """Интегрирование при заданном наклоне s: возвращает x и пары (y, p)."""
    h = 1.0/n
    x = np.linspace(0.0, 1.0, n + 1)
    state = np.empty((n + 1, 2))
    state[0] = [0.0, s]

    for i in range(n):
        u = state[i]
        k1 = rhs(u, a)
        k2 = rhs(u + h*k1/2, a)
        k3 = rhs(u + h*k2/2, a)
        k4 = rhs(u + h*k3, a)
        state[i + 1] = u + h*(k1 + 2*k2 + 2*k3 + k4)/6

    return x, state

## 2. Метод стрельбы

Левое граничное условие выполняется при любом $s$. Для выполнения правого условия нужно найти корень функции
$$F_h(s)=y_h(1;s).$$
Индекс $h$ подчёркивает, что значение на правой опоре вычисляется численно.

Начальный отрезок для поиска: $s\in[-1,0]$. При $s=-1$ конец полученного профиля расположен ниже правой опоры, а при $s=0$ — выше. Корень ищем **методом бисекции**: делим отрезок пополам и сохраняем ту половину, на концах которой невязка меняет знак.

Критерий остановки — $|F_h(s)|\le 10^{-14}$. Он выбран значительно меньше ожидаемых ошибок профиля. Максимальное число делений отрезка — 80. Точное решение при подборе наклона не используется.

In [ ]:
def shoot(n, a=1.0, tol=1e-14, max_iter=80):
    left, right = -1.0, 0.0
    f_left = integrate(left, n, a)[1][-1, 0]
    f_right = integrate(right, n, a)[1][-1, 0]
    if f_left*f_right >= 0:
        raise ValueError("На концах отрезка должны быть разные знаки невязки.")

    history = []
    for iteration in range(1, max_iter + 1):
        s = (left + right)/2
        x, state = integrate(s, n, a)
        residual = state[-1, 0]
        history.append((s, residual))

        if abs(residual) <= tol:
            return {
                "n": n, "h": 1.0/n, "s": s, "x": x, "state": state,
                "residual": residual, "iterations": iteration,
                "history": np.array(history),
            }

        if f_left*residual < 0:
            right = s
        else:
            left = s
            f_left = residual

    raise RuntimeError("Требуемая невязка не достигнута.")


runs = {int(n): shoot(int(n), a, shoot_tol) for n in counts}
main = runs[40]

for s in [-1.0, 0.0]:
    residual = integrate(s, main["n"], a)[1][-1, 0]
    print(f"Начальный наклон s = {s:4.1f}: F_h(s) = {residual:+.8f}")
print(f"\nДля N = {main['n']}, h = {main['h']:.3f}:")
print(f"Найденный наклон: s_h = {main['s']:.12f}")
print(f"Невязка на правой опоре: F_h(s_h) = {main['residual']:+.3e}")
print(f"Число итераций бисекции: {main['iterations']}")

In [ ]:
trial_slopes = [-1.0, -0.75, -0.5, -0.25, 0.0]
fig, ax = plt.subplots(figsize=(9, 5.4), constrained_layout=True)

for s, color in zip(trial_slopes, colors):
    x, state = integrate(s, main["n"], a)
    y = state[:, 0]
    ax.plot(x, y, color=color, lw=1.5, label=f"s = {s:g}")
    ax.plot(1.0, y[-1], "o", ms=5, color=color, mfc="white")

ax.plot(main["x"], main["state"][:, 0], color="#222222", lw=2.2,
        label=f"Найдено: s ≈ {main['s']:.6f}")
ax.scatter([0, 1], [0, 0], marker="s", s=55, color="black",
           label="Опоры", zorder=5)
ax.axhline(0, color="gray", lw=0.8, alpha=0.5)
ax.set(xlabel="x", ylabel="y", xlim=(-0.04, 1.04),
       title="Пристрелочные профили при h = 0.025")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
plt.show()

Пустые кружки показывают правые концы пробных профилей. Чёрный профиль получен после подбора наклона и проходит через обе опоры с заданной численной точностью.

## 3. Сравнение с точным решением

При $a=1$ точный профиль из условия:
$$
y_*(x)=\cosh\left(x-\frac12\right)-\cosh\left(\frac12\right).
$$
Действительно, $y_*'=\sinh(x-\frac12)$, а
$$
y_*''=\cosh\left(x-\frac12\right)
=\sqrt{1+\sinh^2\left(x-\frac12\right)}.
$$
Оба граничных условия также выполнены.

Из точного профиля получаем контрольные значения:
$$
s_*=-\sinh\left(\frac12\right),\qquad
x_{\min}=\frac12,\qquad
y_{\min}=1-\cosh\left(\frac12\right).
$$
Глубина провисания относительно линии опор равна $d=-y_{\min}>0$.

In [ ]:
def exact(x):
    return np.cosh(np.asarray(x) - 0.5) - np.cosh(0.5)


s_exact = -np.sinh(0.5)
sag_exact = np.cosh(0.5) - 1.0
best = runs[int(counts[-1])]
middle_index = best["n"]//2
sag_numeric = -best["state"][middle_index, 0]

print(f"Точный начальный наклон:     {s_exact:.12f}")
print(f"Наклон при N = {best['n']}:         {best['s']:.12f}")
print(f"Ошибка наклона:              {abs(best['s'] - s_exact):.3e}")
print(f"\nТочное провисание:           {sag_exact:.12f}")
print(f"Провисание при N = {best['n']}:     {sag_numeric:.12f}")

x_dense = np.linspace(0.0, 1.0, 1001)
fig, ax = plt.subplots(figsize=(10, 3.6), constrained_layout=True)
ax.plot(x_dense, exact(x_dense), color="#333333", lw=2,
        label="Точная цепная линия")
ax.plot(main["x"], main["state"][:, 0], "o", color=colors[3],
        ms=4, mfc="white", label=f"Стрельба + RK4, h = {main['h']:g}")
ax.scatter([0, 1], [0, 0], marker="s", s=55, color="black", zorder=5,
           label="Опоры")
ax.axhline(0, color="gray", lw=0.8, ls="--")
ax.annotate(f"d ≈ {sag_exact:.6f}",
            xy=(0.5, -sag_exact), xytext=(0.61, -0.075),
            arrowprops={"arrowstyle": "->", "color": "#555555"})
ax.set(xlabel="x", ylabel="y", xlim=(-0.04, 1.04), ylim=(-0.165, 0.055),
       title="Найденная форма нити и точное решение")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=3, fontsize=9)
plt.show()

На масштабе всей нити численные точки практически совпадают с точной кривой. Поэтому точность далее оценивается по разности решений.

## 4. Уменьшение шага и оценка точности

Для каждой сетки наклон подбирается **заново**, с одним и тем же допуском бисекции. Рассматриваются $N=10,\ 20,\ 40,\ 80,\ 160$, то есть $h=0.1,\ 0.05,\ 0.025,\ 0.0125,\ 0.00625$.

Отдельно вычисляются:

- невязка на правой опоре $R_h=|y_h(1)|$;
- ошибка профиля в каждом узле $e_h(x_i)=y_h(x_i)-y_*(x_i)$;
- максимальная узловая ошибка $E_h=\max_i|e_h(x_i)|$.

Максимум берётся по узлам расчётной сетки. Малое $R_h$ показывает точность решения уравнения стрельбы на выбранной сетке, но не заменяет оценку $E_h$.

Экспериментальный порядок при уменьшении шага вдвое:
$$
p_h=\log_2\frac{E_h}{E_{h/2}}.
$$
Если $E_h\approx C h^4$, то ошибка при делении шага пополам уменьшается приблизительно в 16 раз, а $p_h$ близок к 4.

In [ ]:
steps = 1.0/counts
max_errors = []
boundary_errors = []

for n in counts:
    run = runs[int(n)]
    run["error"] = run["state"][:, 0] - exact(run["x"])
    run["max_error"] = np.max(np.abs(run["error"]))
    max_errors.append(run["max_error"])
    boundary_errors.append(abs(run["residual"]))

max_errors = np.array(max_errors)
boundary_errors = np.array(boundary_errors)
orders = np.log2(max_errors[:-1]/max_errors[1:])

print(f"{'N':>5} {'h':>9} {'s_h':>17} {'R_h':>12} {'E_h':>12} {'Порядок':>9}")
for i, n in enumerate(counts):
    run = runs[int(n)]
    order_text = "—" if i == 0 else f"{orders[i-1]:.3f}"
    print(f"{int(n):5d} {run['h']:9.5f} {run['s']:17.12f} "
          f"{boundary_errors[i]:12.3e} {max_errors[i]:12.3e} {order_text:>9}")

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.5), constrained_layout=True)
for n, color in zip(counts, colors):
    run = runs[int(n)]
    error = np.abs(run["error"])
    # Нулевую ошибку нельзя отобразить на логарифмической оси.
    visible_error = np.where(error > 0, error, np.nan)
    ax.semilogy(run["x"], visible_error, color=color, lw=1.6,
                label=f"h = {run['h']:g}")

ax.set(xlabel="x", ylabel=r"$|y_h(x_i)-y_*(x_i)|$",
       title="Ошибка профиля на пяти сетках", xlim=(0, 1))
ax.grid(which="both", alpha=0.2)
ax.legend(ncol=2, fontsize=9)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 4.8), constrained_layout=True)
ax.loglog(steps, max_errors, "o-", color=colors[3], lw=1.8,
          label=r"Максимальная ошибка профиля $E_h$")
ax.loglog(steps, boundary_errors, "s-", color=colors[0], lw=1.2,
          label=r"Невязка на опоре $R_h$")
reference = max_errors[0]*(steps/steps[0])**4
ax.loglog(steps, reference, "--", color="#333333", alpha=0.75,
          label=r"Ориентир $C h^4$")
ax.axhline(shoot_tol, color="#999999", ls=":", label="Допуск бисекции")
ax.set(xlabel="Шаг h", ylabel="Абсолютная ошибка",
       title="Сходимость при уменьшении шага")
ax.grid(which="both", alpha=0.2)
ax.legend(fontsize=9)
plt.show()

Максимальная ошибка профиля уменьшается примерно в 16 раз при каждом делении шага пополам. Наклон зависимости $E_h$ от $h$ на графике с двумя логарифмическими осями близок к 4:
$$\log E_h\approx\log C+4\log h.$$
Это согласуется с четвёртым порядком метода RK4. Граничная невязка остаётся ниже допуска бисекции и не обязана монотонно уменьшаться вместе с шагом.





## Выводы

1. Методом стрельбы найден начальный наклон $s\approx-0.52109530549$. Полученный профиль симметричен относительно середины пролёта с точностью расчёта и совпадает с точной цепной линией. Минимальная высота равна примерно $-0.127625965206$, глубина провисания — $0.127625965206$ в единицах расстояния между опорами.
2. При уменьшении шага с $0.1$ до $0.00625$ максимальная узловая ошибка профиля уменьшается примерно с $1.97\cdot10^{-8}$ до $3.09\cdot10^{-13}$. Экспериментальные порядки близки к 4, что соответствует методу RK4.
3. На всех сетках невязка на правой опоре меньше $10^{-14}$. При этом на грубой сетке ошибка внутри пролёта существенно больше. Поэтому точность подбора начального наклона и точность интегрирования нужно контролировать раздельно.